In [7]:
import ssdeep
import tlsh
import os

#Nos fichiers
file_A = "document_original.txt"
file_B = "document_modifie.txt"   
file_C = "document_different.txt" 

print(f"Fichiers : {file_A}, {file_B}, {file_C}")
print("-" * 50)

# --- ÉTAPE 2 : Fonctions de Hachage  ---

def get_ssdeep_hash(filepath):
    """Calcule le hash SSDEEP d'un fichier."""
    try:
        return ssdeep.hash_from_file(filepath)
    except (AttributeError, NameError):
        # Fallback si hash_from_file n'existe pas dans cette version
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            return ssdeep.hash(f.read())

def get_tlsh_hash(filepath):
    """Calcule le hash TLSH (Ici lecture binaire obligatoire)."""
    with open(filepath, 'rb') as f:
        data = f.read()
        # Parce que TLSH necessite un minimum de donnees (> 50 octets)
        if len(data) < 50:
            return None 
        return tlsh.hash(data)

#--- Exécution et Comparaison ---

# Comparaison A vs B (Ici Modification legitime)
print(f"TEST 1 : Comparaison '{file_A}' vs '{file_B}'")

# SSDEEP
h1 = get_ssdeep_hash(file_A)
h2 = get_ssdeep_hash(file_B)
print(f"[SSDEEP] Hash A : {h1}")
print(f"[SSDEEP] Hash B : {h2}")
score = ssdeep.compare(h1, h2)
print(f"Score SSDEEP : {score}/100") 

# TLSH
t1 = get_tlsh_hash(file_A)
t2 = get_tlsh_hash(file_B)

if t1 and t2:
    print(f"[TLSH]   Hash A : {t1}")
    print(f"[TLSH]   Hash B : {t2}")
    distance = tlsh.diff(t1, t2)
    print(f"Distance TLSH : {distance} ") #Plus c'est bas mieux c'est
else:
    print("[TLSH] Erreur : Fichier<50 octets).")

print("\n" + "="*50 + "\n")

# Comparaison A vs C (Deux Fichiers qui sont completement differents)
print(f"TEST 2 : Comparaison '{file_A}' vs '{file_C}'")

h3 = get_ssdeep_hash(file_C)
score_diff = ssdeep.compare(h1, h3)
print(f"Score SSDEEP  : {score_diff}") #Normalement on doit trouver 0

t3 = get_tlsh_hash(file_C)
if t1 and t3:
    dist_diff = tlsh.diff(t1, t3)
    print(f"Distance TLSH  : {dist_diff}") #(On doit trouver >100)

Fichiers : document_original.txt, document_modifie.txt, document_different.txt
--------------------------------------------------
TEST 1 : Comparaison 'document_original.txt' vs 'document_modifie.txt'
[SSDEEP] Hash A : 6:q7+/ev57EuHhEsk4rGktxkiMDWg5hakLk8PRbgX:TBMg4rjBMDWg5hvLk46X
[SSDEEP] Hash B : 6:q7+/ev57EuHhEsk4rGktxkiMDWg5hakLk8PRbgP4RkQr:TBMg4rjBMDWg5hvLk46gRp
Score SSDEEP : 80/100
[TLSH]   Hash A : T1C5D0970C0A252AE0A38990C803B04F88C5422198BA312AC8721758AB10A0E8AF946410
[TLSH]   Hash B : T17AD02B4E19112AE0A3D5F4D817B05A89C58221997E3625C8B217ACE65594EC9BA55810
Distance TLSH : 157 


TEST 2 : Comparaison 'document_original.txt' vs 'document_different.txt'
Score SSDEEP  : 0
Distance TLSH  : 373


In [6]:
from PIL import Image
import imagehash
import os

# --- Configuration des fichiers ---
img_original   = "original_168x300.png"
img_modifie    = "modified_160x295.png"  # Fichier Redimensionn
img_compresse  = "compressed.png"        # Fichier compresse

print(f"Chargement des images : \n 1. {img_original}\n 2. {img_modifie}\n 3. {img_compresse}")
print("-" * 50)

# --- Calcul du Perceptual Hash (pHash) ---
try:
    hash_orig = imagehash.phash(Image.open(img_original))
    hash_mod  = imagehash.phash(Image.open(img_modifie))
    hash_comp = imagehash.phash(Image.open(img_compresse))
except FileNotFoundError as e:
    print(f"Erreur :Ne trouve pas les images ({e})")
    exit()

print(f"Empreinte (Hash) Originale  : {hash_orig}")
print(f"Empreinte (Hash) Modifiée   : {hash_mod}")
print(f"Empreinte (Hash) Compressée : {hash_comp}")
print("-" * 50)

# --- Analyse de la Similarité ( avec distance Distance de Hamming) ---
# On soustrait simplement les hashs pour obtenir le nombre de bits différents.

dist_mod = hash_orig - hash_mod
dist_comp = hash_orig - hash_comp

def analyser_resultat(nom_test, distance):
    print(f"TEST : Original vs {nom_test}")
    print(f"Distance : {distance}")
    
    # Treshold pour pHash : <= 10
    if distance == 0:
        print("=> RÉSULTAT : Identique ")
    elif distance <= 10:
        print("=> RÉSULTAT : SUCCÈS - Modification légitime détectée.")
        print("   (L'image est presque identique même malgré les changements de fichiers)")
    else:
        print("=> RÉSULTAT : ÉCHEC - Images trop différentes.")
    print("-" * 30)

analyser_resultat("Modifiée (Resize)", dist_mod)
analyser_resultat("Compressée", dist_comp)

Chargement des images : 
 1. original_168x300.png
 2. modified_160x295.png
 3. compressed.png
--------------------------------------------------
Empreinte (Hash) Originale  : 8763789c64b179a3
Empreinte (Hash) Modifiée   : 8763789c64396b96
Empreinte (Hash) Compressée : 8763789c64b179a3
--------------------------------------------------
TEST : Original vs Modifiée (Resize)
Distance : 8
=> RÉSULTAT : SUCCÈS - Modification légitime détectée.
   (L'image est presque identique même malgré les changements de fichiers)
------------------------------
TEST : Original vs Compressée
Distance : 0
=> RÉSULTAT : Identique 
------------------------------
